# Web3Geeks Machine Learning Internship — Week 1 Day 1
## Project: UCI Census Income Classification (>50K / <=50K)

**Author:** Armish Iqbal  
**Repository:** [armishiqbal/internship_web3geeks](https://github.com/armishiqbal/internship_web3geeks)  
**Objective:** Build a disciplined, end-to-end Machine Learning foundation for Census Income prediction, including exploratory data analysis, reproducible stratified splitting, baseline benchmarking, and deep error analysis.

---
### 📊 Dataset Scope Note
The canonical UCI Adult Census dataset comprises $32,561$ records in `adult.data` (and $16,281$ records in `adult.test`, totaling $48,842$ across the full corpus). In this notebook, we utilize the canonical $32,561$-row dataset with a strict 3-way stratified partition ($70\%$ Train / $10\%$ Dev / $20\%$ Test).

## 1. Environment Setup & Imports
We import data manipulation, visualization, and statistical modeling libraries.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

# Visualization aesthetics
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'

print("Libraries successfully loaded.")

## 2. Task 1: Problem Definition, Business Objective & Class Base Rate

### Business Context & Target Definition
* **Target Variable:** Binary income classification ($y=1$ if Income $>\$50\text{K}$, $y=0$ if $\le\$50\text{K}$).
* **Business Use Case:** Identifying affluent individuals for high-touch financial advisory and wealth management outreach.
* **Cost Trade-off:** Direct customer acquisition (phone consultation, personalized mail) carries high variable costs. Contacting false positives ($y=0$ predicted as $y=1$) wastes expensive sales capacity. Therefore, **Precision** (and **$F_1$-Score**) is our primary evaluation metric rather than unweighted Accuracy.

In [ ]:
COLUMNS = [
    'age', 'workclass', 'fnlwgt', 'education', 'education_num',
    'marital_status', 'occupation', 'relationship', 'race', 'sex',
    'capital_gain', 'capital_loss', 'hours_per_week', 'native_country',
    'income'
]

csv_path = 'adults.csv'
df = pd.read_csv(
    csv_path,
    names=COLUMNS,
    na_values=['?', ' ?', '? '],
    skipinitialspace=True
)

# Clean target string into binary indicator
df['target'] = (df['income'].astype(str).str.strip().str.replace('.', '', regex=False) == '>50K').astype(int)

total_records = len(df)
positive_records = df['target'].sum()
negative_records = total_records - positive_records
base_rate = df['target'].mean() * 100

print(f"Total Dataset Rows   : {total_records:,}")
print(f"Positive Class (>50K): {positive_records:,} ({base_rate:.2f}%)")
print(f"Negative Class (<=50K): {negative_records:,} ({100-base_rate:.2f}%)")
print(f"Base Rate / Prevalence: {base_rate:.2f}%")

## 3. Task 2: Data Cleaning & Exploratory Data Analysis (EDA)

We inspect missing value profiles, summary statistics for numerical features, and calculate conditional high-income rates across demographic segments.

In [ ]:
# Missing Value Inspection
missing = df.isna().sum()
missing_pct = (missing / len(df)) * 100
missing_df = pd.DataFrame({'Missing_Count': missing, 'Missing_Pct (%)': missing_pct.round(2)})
print("Missing Values per Column:")
display(missing_df[missing_df['Missing_Count'] > 0])

In [ ]:
# Education Level vs High-Income Rate (>50K)
edu_summary = df.groupby('education', observed=False).agg(
    Total_Count=('target', 'count'),
    High_Income_Count=('target', 'sum'),
    High_Income_Rate=('target', lambda x: f"{(x.mean() * 100):.2f}%")
).sort_values(by='Total_Count', ascending=False)

print("Demographic Summary: Education Level vs >50K Rate")
display(edu_summary)

In [ ]:
# 4-Panel EDA Visualizations
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Plot 1: Target Imbalance
sns.countplot(
    data=df, x='target', hue='target', palette=['#4C72B0', '#55A868'],
    ax=axes[0, 0], legend=False
)
axes[0, 0].set_title('1. Target Distribution (0: <=50K, 1: >50K)', fontsize=13, fontweight='bold')
axes[0, 0].set_xticklabels(['<=50K', '>50K'])
axes[0, 0].set_ylabel('Count')

# Plot 2: Age Distribution by Income
sns.histplot(
    data=df, x='age', hue='target', multiple='stack', bins=30,
    palette=['#4C72B0', '#C44E52'], ax=axes[0, 1]
)
axes[0, 1].set_title('2. Age Distribution by Income Class', fontsize=13, fontweight='bold')
axes[0, 1].set_xlabel('Age (Years)')

# Plot 3: High-Income Rate by Education
edu_order = df.groupby('education', observed=False)['target'].mean().sort_values(ascending=False).index
sns.barplot(
    data=df, x='education', y='target', order=edu_order,
    palette='Blues_r', ax=axes[1, 0], errorbar=None, hue='education', legend=False
)
axes[1, 0].set_title('3. High-Income Rate (>50K) by Education Level', fontsize=13, fontweight='bold')
axes[1, 0].set_ylabel('Proportion Earning >50K')
axes[1, 0].tick_params(axis='x', rotation=50)

# Plot 4: Hours Worked per Week
sns.histplot(
    data=df, x='hours_per_week', hue='target', multiple='layer', bins=30,
    palette=['#4C72B0', '#DD8452'], ax=axes[1, 1], alpha=0.6
)
axes[1, 1].set_title('4. Hours Worked per Week by Income Class', fontsize=13, fontweight='bold')
axes[1, 1].set_xlabel('Hours per Week')

plt.tight_layout()
plt.show()

## 4. Task 3: Reproducible Stratified Splits (70 / 10 / 20)

We partition the dataset into:
1. **Train Set (70%):** $22,792$ rows for model training.
2. **Dev / Validation Set (10%):** $3,256$ rows for hyperparameter tuning and model selection.
3. **Hold-Out Test Set (20%):** $6,513$ rows for strictly unbiased final benchmarking.

> **Why a Hold-Out Test Set is Critical:**  
> Tuning hyperparameters or feature transforms directly against the test set causes **data snooping / test leakage**. The model inadvertently learns noise specific to that split, resulting in over-optimistic performance estimates that fail to generalize in production.

In [ ]:
X = df.drop(columns=['income', 'target'])
y = df['target']

RANDOM_STATE = 42

# Step 1: Hold out 20% for test
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

# Step 2: Split remaining 80% into 70% train and 10% dev (0.125 * 0.80 = 0.10)
X_train, X_dev, y_train, y_dev = train_test_split(
    X_train_val, y_train_val, test_size=0.125, random_state=RANDOM_STATE, stratify=y_train_val
)

splits_data = [
    {"Split": "Full Dataset", "Rows": len(df), "Positive Count": df['target'].sum(), "Positive Rate (%)": f"{df['target'].mean()*100:.2f}%", "Split Share (%)": "100.0%"},
    {"Split": "Train Set (70%)", "Rows": len(X_train), "Positive Count": y_train.sum(), "Positive Rate (%)": f"{y_train.mean()*100:.2f}%", "Split Share (%)": f"{(len(X_train)/len(df))*100:.1f}%"},
    {"Split": "Dev Set (10%)", "Rows": len(X_dev), "Positive Count": y_dev.sum(), "Positive Rate (%)": f"{y_dev.mean()*100:.2f}%", "Split Share (%)": f"{(len(X_dev)/len(df))*100:.1f}%"},
    {"Split": "Test Set (20%)", "Rows": len(X_test), "Positive Count": y_test.sum(), "Positive Rate (%)": f"{y_test.mean()*100:.2f}%", "Split Share (%)": f"{(len(X_test)/len(df))*100:.1f}%"},
]

split_df = pd.DataFrame(splits_data)
display(split_df)

## 5. Task 4: Baseline Models & Metric Evaluation

We benchmark two non-trivial baselines against the $6,513$-instance hold-out test set:
1. **Majority-Class Baseline:** Predicts the most frequent class (always $\le\$50\text{K}$).
2. **Single-Feature Rule-Based Baseline:** Predicts $>\$50\text{K}$ whenever `education_num >= 13` (Bachelors, Masters, Professional School, Doctorate).

In [ ]:
# Baseline 1: Majority-Class Classifier
majority_clf = DummyClassifier(strategy='most_frequent')
majority_clf.fit(X_train, y_train)
y_pred_maj = majority_clf.predict(X_test)
y_prob_maj = np.full(len(y_test), y_train.mean())

# Baseline 2: Single-Feature Heuristic (education_num >= 13)
y_pred_rule = (X_test['education_num'] >= 13).astype(int)
y_prob_rule = (X_test['education_num'] >= 13).astype(float)

def evaluate_predictions(y_true, y_pred, y_prob):
    return {
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred, zero_division=0),
        'Recall': recall_score(y_true, y_pred, zero_division=0),
        'F1-Score': f1_score(y_true, y_pred, zero_division=0),
        'ROC-AUC': roc_auc_score(y_true, y_prob),
        'PR-AUC': average_precision_score(y_true, y_prob),
        'CM': confusion_matrix(y_true, y_pred)
    }

eval_maj = evaluate_predictions(y_test, y_pred_maj, y_prob_maj)
eval_rule = evaluate_predictions(y_test, y_pred_rule, y_prob_rule)

summary_metrics = pd.DataFrame([
    {
        'Baseline Model': '1. Majority-Class (Always <=50K)',
        'Accuracy': f"{eval_maj['Accuracy']:.4f}",
        'Precision': f"{eval_maj['Precision']:.4f}",
        'Recall': f"{eval_maj['Recall']:.4f}",
        'F1-Score': f"{eval_maj['F1-Score']:.4f}",
        'ROC-AUC': f"{eval_maj['ROC-AUC']:.4f}",
        'PR-AUC': f"{eval_maj['PR-AUC']:.4f}"
    },
    {
        'Baseline Model': '2. Single-Feature Rule (education_num >= 13)',
        'Accuracy': f"{eval_rule['Accuracy']:.4f}",
        'Precision': f"{eval_rule['Precision']:.4f}",
        'Recall': f"{eval_rule['Recall']:.4f}",
        'F1-Score': f"{eval_rule['F1-Score']:.4f}",
        'ROC-AUC': f"{eval_rule['ROC-AUC']:.4f}",
        'PR-AUC': f"{eval_rule['PR-AUC']:.4f}"
    }
])

print("--- BASELINE PERFORMANCE COMPARISON ---")
display(summary_metrics)

print("\nConfusion Matrix — Single-Feature Rule:")
cm = eval_rule['CM']
print(f"  True Negatives  (TN): {cm[0, 0]:<5} | False Positives (FP): {cm[0, 1]:<5}")
print(f"  False Negatives (FN): {cm[1, 0]:<5} | True Positives  (TP): {cm[1, 1]:<5}")

### 📈 Interpretation & ML Utility Criteria
1. **Why the Heuristic Beats Majority Prediction:**  
   Although the majority class predictor attains $75.91\%$ accuracy, it produces **$0.00$ Precision and $0.00$ Recall** for identifying $>\$50\text{K}$ earners. In contrast, the single-feature education rule isolates genuine signal, capturing **$47.46\%$ Precision**, **$48.79\%$ Recall**, and an **$F_1$-Score of $0.4811$** ($\text{PR-AUC} = 0.3548$).

2. **Minimum Threshold for Useful ML:**  
   To justify engineering and deployment complexity over a 1-line rule, an ML model must achieve **$F_1 > 0.65$** (a $>35\%$ relative improvement), $\text{PR-AUC} > 0.60$, and Precision $\ge 75\%$ at $\ge 60\%$ Recall.

## 6. Task 5: Error Analysis & Day 2 Feature Engineering Roadmap

We inspect the False Positives ($FP=847$) and False Negatives ($FN=804$) from the baseline to uncover structural failure modes.

In [ ]:
test_analysis = X_test.copy()
test_analysis['actual'] = y_test
test_analysis['predicted'] = (test_analysis['education_num'] >= 13).astype(int)

fp_df = test_analysis[(test_analysis['actual'] == 0) & (test_analysis['predicted'] == 1)]
fn_df = test_analysis[(test_analysis['actual'] == 1) & (test_analysis['predicted'] == 0)]

print(f"Total Test Samples:    {len(test_analysis):,}")
print(f"False Positives (FP):  {len(fp_df):,} ({len(fp_df)/len(test_analysis)*100:.2f}% of test set)")
print(f"False Negatives (FN):  {len(fn_df):,} ({len(fn_df)/len(test_analysis)*100:.2f}% of test set)")

cols_to_inspect = ['age', 'education', 'education_num', 'marital_status', 'occupation', 'hours_per_week', 'capital_gain']

print("\n--- Sample False Positives (Degree Holders Earning <=50K) ---")
display(fp_df[cols_to_inspect].sample(n=5, random_state=42))

print("\n--- Sample False Negatives (Non-Degree Holders Earning >50K) ---")
display(fn_df[cols_to_inspect].sample(n=5, random_state=42))

### 🛠️ Key Error Diagnoses & Concrete Day 2 Fixes

| Error Mode | Root Cause Observed in Data | Actionable Day 2 Feature Engineering Solution |
| :--- | :--- | :--- |
| **False Positives** | Young degree holders (age $<30$) & part-time workers ($<35$ hrs) | **Seniority interactions:** Create `age * education_num` and `hours_per_week * education_num` |
| **False Negatives** | Experienced trades/craftsmen earning overtime without degrees | **Overtime & Experience flags:** Add `is_overtime (>40 hrs)` and `work_experience (age - education_num - 6)` |
| **Capital Income** | Extreme right-skewed capital gains ($>90\%$ zeros) | **Binary flags & log transforms:** `has_capital_gain` indicator + `log1p(capital_gain)` |
| **Missing Values** | Missing entries in `workclass`, `occupation`, `native_country` | **Explicit token:** Impute with `'Missing'` / `'Unknown'` category so tree models capture signal |
| **Household Status** | `Married-civ-spouse` is a dominant high-income predictor | **Consolidated Marital Status:** Group into `is_married_spouse_present`, `is_single`, `is_divorced` |

## 7. Deliverable Summary & Stakeholder Takeaways

1. **Base Rate:** The target class ($>\$50\text{K}$) occurs at **$24.08\%$** in the general adult census population.
2. **Stratification:** Successfully implemented strict 70/10/20 train/dev/test partitioning with zero data leakage.
3. **Baseline Benchmark:** Single-feature rule achieves **Precision: $0.4746$**, **Recall: $0.4879$**, and **$F_1$: $0.4811$**.
4. **Optimization Target:** All upcoming Machine Learning models (Logistic Regression, Random Forest, LightGBM/XGBoost) will optimize for **$F_1$-Score / PR-AUC**, targeting $F_1 > 0.65$ with Precision $> 75\%$.